# Solve PyPSA-Eur with LCA constraints, and analyse the results

The LCA constraints are in `lca_constraints.py`, which PyPSA-Eur calls in its solving rules when its configuration
contains (paths relative to the `rules` folder):

```yaml
solving:
  options:
    custom_extra_functionality: ../lca/lca_constraints.py
```

Their settings (input files, limits, objective, CO2 limits of PyPSA-Eur) are in `lca_settings.yaml`. The network is
solved by Snakemake, as any PyPSA-Eur run (below, or from the pixi shell), and analyzed with the same code
(`LCA.results()`), so that the notebook and the constraints never diverge.

In [1]:
import subprocess
import sys

import pandas as pd
import pypsa
import yaml

from lca_constraints import LCA

CONFIG = 'config/test/config.perfect-lca.yaml'  # PyPSA-Eur configuration, with custom_extra_functionality (see above)
RUN = 'test-sector-perfect'  # run name of the configuration
HORIZON = 2050  # last planning horizon (perfect foresight: the network that contains all years)
SOLVE = False  # True: solve with Snakemake from this notebook (otherwise, from the pixi shell, same command)
CLIMATE_CHANGE = 'm_CCS_tot'  # indicator compared with PyPSA's CO2 emissions

## Solve with Snakemake

In [2]:
command = [sys.executable, '-m', 'snakemake', '-call', f'results/{RUN}/networks/solved_{HORIZON}.nc',
           '--configfile', CONFIG,
           '--forcerun', 'solve_network',  # re-solve: lca_settings.yaml is not an input of the rule
           '--rerun-triggers', 'mtime']  # do not rebuild the composed network because code or config files changed
print(' '.join(command[2:]))
if SOLVE:
    run = subprocess.run(command, cwd='..', capture_output=True, text=True)
    lines = run.stderr.splitlines()
    errors = [i for i, line in enumerate(lines) if line.startswith('Error in rule')]
    print('\n'.join(lines[errors[0]:errors[0] + 30]) if errors else '\n'.join(lines[-30:]))  # error, or end of the log
    assert run.returncode == 0, 'Snakemake failed: see its messages above'

snakemake -call results/test-sector-perfect/networks/solved_2050.nc --configfile config/test/config.perfect-lca.yaml --forcerun solve_network --rerun-triggers mtime


## Results

For each technology, year, part (infrastructure, operation, direct emissions) and impact category: capacity (active in
that year) or yearly dispatch, impact per unit of capacity or dispatch, and yearly impact.

In [3]:
n = pypsa.Network(f'../results/{RUN}/networks/solved_{HORIZON}.nc')
settings = yaml.safe_load(open('lca_settings.yaml'))
units = pd.read_csv(settings['abbrev']).drop_duplicates('Abbrev').set_index('Abbrev')['Unit']

results = LCA(n, settings).results()
results['Impact category unit'] = results['Impact category'].map(units)
results.to_csv('results/lca_results.csv', index=False)

summary = results.pivot_table(index=['Impact category', 'Type'], columns='Year', values='Yearly impact', aggfunc='sum')
summary['Unit'] = summary.index.get_level_values('Impact category').map(units)
print(f'Total system cost (objective): {n.objective:.4e}')
summary.loc['m_CCS_tot']

INFO:pypsa.network.io:New version 1.4.0 available! (Current: 1.2.4)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores, sub_networks
C:\Users\matth\PycharmProjects\pypsa-eur-lca\lca\lca_constraints.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sign = np.where((tech['Component'] == 'Link') & tech['Carrier'].map(backwards).fillna(False).astype(bool), -1, 1)


Total system cost (objective): 1.2732e+10


Year,2030,2040,2050,Unit
Type,,,,
Direct,2.973203e+09,2.184680e+09,-1.937151e-06,kg CO2 eq (short)
Infrastructure,1.843908e+10,1.872887e+10,1.910614e+10,kg CO2 eq (short)
Operation,3.597695e+09,2.532954e+09,3.661120e+08,kg CO2 eq (short)


In [4]:
# Contributions of each technology to CLIMATE_CHANGE in the last year, next to PyPSA's CO2 emissions (balance of the co2 bus)
year = results['Year'].max()
climate = results[(results['Impact category'] == CLIMATE_CHANGE) & (results['Year'] == year)]
comparison = climate.pivot_table(index=['Component', 'Carrier'], columns='Type', values='Yearly impact', aggfunc='sum') / 1e9
comparison.columns = [f'{CLIMATE_CHANGE} {part} (Mt CO2 eq/yr)' for part in comparison.columns]

co2 = n.statistics.energy_balance(bus_carrier='co2', groupby='carrier', nice_names=False)
co2 = co2[year] if isinstance(co2, pd.DataFrame) else co2
co2 = co2.drop(('Store', 'co2'), errors='ignore') / 1e6
comparison = comparison.join(co2.rename('PyPSA CO2 emissions (Mt CO2/yr)').rename_axis(['Component', 'Carrier']), how='outer')
comparison = comparison.fillna(0)
comparison = comparison[(comparison.abs() > 1e-6).any(axis=1)]
comparison.loc[('Total', ''), :] = comparison.sum()
comparison.loc['Total'].round(3)

,m_CCS_tot Direct (Mt CO2 eq/yr),m_CCS_tot Infrastructure (Mt CO2 eq/yr),m_CCS_tot Operation (Mt CO2 eq/yr),PyPSA CO2 emissions (Mt CO2/yr)
Carrier,,,,
,-0.0,19.106,0.366,-0.538
